# Kronos Time-Series Forecasting on Google Colab
## Jasper Trades - Remote AI Model Server

This notebook runs **three Kronos models concurrently** on Google Colab's free GPU:
- **Kronos-mini** (2k tokenizer, 2048 context) - Fast screener
- **Kronos-small** (base tokenizer, 512 context) - Balanced accuracy
- **Kronos-base** (base tokenizer, 512 context) - Highest accuracy

It exposes a REST API that your local Jasper Trades backend can connect to.

**Architecture:**
- All 3 models loaded simultaneously (~600MB total)
- Cascade filtering: mini → small → base for efficiency
- Context-length routing: auto-select model based on data length
- Ensemble mode: average predictions for robustness

**Memory Requirements:**
- System RAM: ~2GB (Colab free tier: 12.7GB)
- GPU VRAM: ~1GB (Colab free tier: ~15GB)

**Important:** Keep this tab open and disable browser sleep extensions.

## Cell 1: Install Dependencies
Run this first (takes 2-3 minutes)

In [ ]:
# Install dependencies
!pip install torch transformers accelerate fastapi uvicorn nest-asyncio yfinance pandas numpy pyqlib

# Clone Kronos repo to get the model module (required for NeoQuasar models)
!git clone --depth 1 https://github.com/shiyu-coder/Kronos.git /content/Kronos

# Add the model directory to Python path
import sys
sys.path.insert(0, '/content/Kronos')

# Verify the model module is available
try:
    from model import Kronos, KronosTokenizer, KronosPredictor
    print("✅ Kronos model module imported successfully!")
    print(f"   Available classes: Kronos, KronosTokenizer, KronosPredictor")
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("   Trying alternative import method...")
    sys.path.insert(0, '/content/Kronos/model')
    from model import Kronos, KronosTokenizer, KronosPredictor
    print("✅ Import successful with alternative path!")

print("\n✅ Dependencies installed and Kronos model module loaded")

# Check GPU availability
import torch
print(f"\n🔥 GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   Device: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Cell 2: Load Kronos Model
Run this after Cell 1 completes (takes 1-2 minutes)

In [ ]:
import torch
import warnings
warnings.filterwarnings('ignore')

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading Kronos models on {device}...")
print("=" * 50)

# Load tokenizers first
print("\n📚 Loading tokenizers...")
tokenizer_base = KronosTokenizer.from_pretrained("NeoQuasar/Kronos-Tokenizer-base")
tokenizer_2k = KronosTokenizer.from_pretrained("NeoQuasar/Kronos-Tokenizer-2k")
print("✅ Tokenizers loaded")

# Load all three models
print("\n🤖 Loading models...")
print("  - Kronos-mini (fast screener)...")
model_mini = Kronos.from_pretrained("NeoQuasar/Kronos-mini")

print("  - Kronos-small (balanced)...")
model_small = Kronos.from_pretrained("NeoQuasar/Kronos-small")

print("  - Kronos-base (highest accuracy)...")
model_base = Kronos.from_pretrained("NeoQuasar/Kronos-base")
print("✅ All models loaded")

# Create predictor instances
print("\n⚙️ Creating predictors...")
predictor_mini = KronosPredictor(model_mini, tokenizer_2k, device=device, max_context=2048)
predictor_small = KronosPredictor(model_small, tokenizer_base, device=device, max_context=512)
predictor_base = KronosPredictor(model_base, tokenizer_base, device=device, max_context=512)

print("✅ All predictors ready")

# Display model info
print("\n" + "=" * 50)
print("📊 Model Configuration:")
print(f"  predictor_mini:  max_context=2048, tokenizer=2k")
print(f"  predictor_small: max_context=512,  tokenizer=base")
print(f"  predictor_base:  max_context=512,  tokenizer=base")
print(f"\n💾 Total memory usage: ~600 MB")
print(f"🔥 Device: {device}")

## Cell 3: Prediction Functions
Run this to define prediction logic

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Tuple

def fetch_price_data(symbol: str, days: int = 30) -> Optional[pd.DataFrame]:
    """Fetch historical price data from Yahoo Finance"""
    try:
        df = yf.download(symbol, period=f"{days}d", interval="1d", progress=False)
        if df.empty:
            return None
        # Flatten column names if multi-level
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        return df.reset_index()
    except Exception as e:
        print(f"Error fetching data for {symbol}: {e}")
        return None

def prepare_kronos_input(df: pd.DataFrame, lookback: int) -> Tuple[pd.DataFrame, pd.Series, pd.Series]:
    """Prepare OHLCV data for Kronos predictor"""
    # Ensure required columns exist
    required_cols = ['open', 'high', 'low', 'close']
    for col in required_cols:
        if col not in df.columns:
            # Try case-insensitive match
            for c in df.columns:
                if c.lower() == col:
                    df[col] = df[c]
                    break
    
    # Generate timestamps if not present
    if 'timestamps' not in df.columns and 'Date' in df.columns:
        df['timestamps'] = pd.to_datetime(df['Date'])
    elif 'timestamp' in df.columns:
        df['timestamps'] = pd.to_datetime(df['timestamp'])
    
    # Use last 'lookback' rows
    df = df.tail(lookback).reset_index(drop=True)
    
    x_df = df[['open', 'high', 'low', 'close']].copy()
    x_timestamp = df['timestamps']
    
    # Create future timestamps for prediction
    last_ts = x_timestamp.iloc[-1]
    y_timestamp = pd.Series([last_ts + timedelta(days=i+1) for i in range(1)])
    
    return x_df, x_timestamp, y_timestamp

# ============================================================
# Strategy 1: Cascade Filtering (High Efficiency)
# ============================================================
def predict_cascade(symbol: str, lookback_days: int = 30) -> Dict:
    """
    Cascade strategy: Use mini as screener, then escalate to base for high-confidence signals.
    """
    start_time = datetime.utcnow()
    
    df = fetch_price_data(symbol, lookback_days)
    if df is None or len(df) < 10:
        return {"symbol": symbol, "direction": "UNKNOWN", "confidence": 0.0, "error": "Insufficient data"}
    
    # Step 1: Run fast mini model as initial screener
    try:
        x_df, x_ts, y_ts = prepare_kronos_input(df, lookback=100)
        mini_result = predictor_mini.predict(df=x_df, x_timestamp=x_ts, y_timestamp=y_ts, pred_len=1, T=1.0, top_p=0.9, sample_count=1)
        mini_pred = mini_result['close'].iloc[-1] if isinstance(mini_result, pd.DataFrame) else mini_result[0]
    except:
        mini_pred = 0
    
    # Filter: if mini shows weak signal, return early
    if abs(mini_pred) < 0.001:
        return {
            "symbol": symbol,
            "direction": "NEUTRAL",
            "confidence": 0.3,
            "strategy": "cascade_filtered_at_mini",
            "timestamp": datetime.utcnow().isoformat(),
            "inference_time_ms": round((datetime.utcnow() - start_time).total_seconds() * 1000)
        }
    
    # Step 2: Escalate to small for moderate confidence
    try:
        x_df, x_ts, y_ts = prepare_kronos_input(df, lookback=50)
        small_result = predictor_small.predict(df=x_df, x_timestamp=x_ts, y_timestamp=y_ts, pred_len=1, T=1.0, top_p=0.9, sample_count=1)
        small_pred = small_result['close'].iloc[-1] if isinstance(small_result, pd.DataFrame) else small_result[0]
    except:
        small_pred = mini_pred
    
    # Filter again
    if abs(small_pred) < 0.002:
        return {
            "symbol": symbol,
            "direction": "NEUTRAL",
            "confidence": 0.4,
            "strategy": "cascade_filtered_at_small",
            "timestamp": datetime.utcnow().isoformat(),
            "inference_time_ms": round((datetime.utcnow() - start_time).total_seconds() * 1000)
        }
    
    # Step 3: Full base model for final prediction
    try:
        x_df, x_ts, y_ts = prepare_kronos_input(df, lookback=50)
        base_result = predictor_base.predict(df=x_df, x_timestamp=x_ts, y_timestamp=y_ts, pred_len=1, T=1.0, top_p=0.9, sample_count=1)
        base_pred = base_result['close'].iloc[-1] if isinstance(base_result, pd.DataFrame) else base_result[0]
    except:
        base_pred = small_pred
    
    direction = "UP" if base_pred > 0 else "DOWN"
    confidence = min(abs(base_pred) * 100, 0.95)
    
    return {
        "symbol": symbol,
        "direction": direction,
        "confidence": round(confidence, 3),
        "predicted_change": round(base_pred, 6),
        "strategy": "cascade_full",
        "timestamp": datetime.utcnow().isoformat(),
        "inference_time_ms": round((datetime.utcnow() - start_time).total_seconds() * 1000)
    }

# ============================================================
# Strategy 2: Context-Length Routing
# ============================================================
def predict_context_routed(symbol: str, lookback_days: int = 60) -> Dict:
    """
    Context-length routing: Auto-select model based on data length.
    """
    start_time = datetime.utcnow()
    
    df = fetch_price_data(symbol, lookback_days)
    if df is None or len(df) < 10:
        return {"symbol": symbol, "direction": "UNKNOWN", "confidence": 0.0, "error": "Insufficient data"}
    
    data_length = len(df)
    
    try:
        if data_length > 512:
            # Route to mini for long context
            print(f"  [Context Routing] {symbol}: {data_length} steps → using mini (2048 context)")
            x_df, x_ts, y_ts = prepare_kronos_input(df, lookback=min(2000, data_length))
            result = predictor_mini.predict(df=x_df, x_timestamp=x_ts, y_timestamp=y_ts, pred_len=1, T=1.0, top_p=0.9, sample_count=1)
            strategy = "context_routed_mini"
        else:
            # Route to base for best accuracy
            print(f"  [Context Routing] {symbol}: {data_length} steps → using base (512 context)")
            x_df, x_ts, y_ts = prepare_kronos_input(df, lookback=50)
            result = predictor_base.predict(df=x_df, x_timestamp=x_ts, y_timestamp=y_ts, pred_len=1, T=1.0, top_p=0.9, sample_count=1)
            strategy = "context_routed_base"
        
        pred = result['close'].iloc[-1] if isinstance(result, pd.DataFrame) else result[0]
    except Exception as e:
        return {
            "symbol": symbol,
            "direction": "ERROR",
            "confidence": 0.0,
            "error": str(e)
        }
    
    direction = "UP" if pred > 0 else "DOWN"
    confidence = min(abs(pred) * 100, 0.95)
    
    return {
        "symbol": symbol,
        "direction": direction,
        "confidence": round(confidence, 3),
        "predicted_change": round(pred, 6),
        "data_length": data_length,
        "strategy": strategy,
        "timestamp": datetime.utcnow().isoformat(),
        "inference_time_ms": round((datetime.utcnow() - start_time).total_seconds() * 1000)
    }

# ============================================================
# Strategy 3: Model Ensembling
# ============================================================
def predict_ensemble(symbol: str, lookback_days: int = 30, weights: Tuple[float, float, float] = (0.2, 0.3, 0.5)) -> Dict:
    """
    Ensemble prediction: Weighted average of all three models.
    """
    start_time = datetime.utcnow()
    
    df = fetch_price_data(symbol, lookback_days)
    if df is None or len(df) < 10:
        return {"symbol": symbol, "direction": "UNKNOWN", "confidence": 0.0, "error": "Insufficient data"}
    
    predictions = []
    
    try:
        # Mini prediction
        x_df, x_ts, y_ts = prepare_kronos_input(df, lookback=200)
        mini_result = predictor_mini.predict(df=x_df, x_timestamp=x_ts, y_timestamp=y_ts, pred_len=1, T=1.0, top_p=0.9, sample_count=1)
        mini_pred = mini_result['close'].iloc[-1] if isinstance(mini_result, pd.DataFrame) else mini_result[0]
        predictions.append(mini_pred * weights[0])
    except:
        pass
    
    try:
        # Small prediction
        x_df, x_ts, y_ts = prepare_kronos_input(df, lookback=50)
        small_result = predictor_small.predict(df=x_df, x_timestamp=x_ts, y_timestamp=y_ts, pred_len=1, T=1.0, top_p=0.9, sample_count=1)
        small_pred = small_result['close'].iloc[-1] if isinstance(small_result, pd.DataFrame) else small_result[0]
        predictions.append(small_pred * weights[1])
    except:
        pass
    
    try:
        # Base prediction
        x_df, x_ts, y_ts = prepare_kronos_input(df, lookback=50)
        base_result = predictor_base.predict(df=x_df, x_timestamp=x_ts, y_timestamp=y_ts, pred_len=1, T=1.0, top_p=0.9, sample_count=1)
        base_pred = base_result['close'].iloc[-1] if isinstance(base_result, pd.DataFrame) else base_result[0]
        predictions.append(base_pred * weights[2])
    except:
        pass
    
    if not predictions:
        return {"symbol": symbol, "direction": "ERROR", "confidence": 0.0, "error": "All models failed"}
    
    # Weighted ensemble prediction
    ensemble_pred = sum(predictions)
    direction = "UP" if ensemble_pred > 0 else "DOWN"
    confidence = min(abs(ensemble_pred) * 100, 0.95)
    
    return {
        "symbol": symbol,
        "direction": direction,
        "confidence": round(confidence, 3),
        "predicted_change": round(ensemble_pred, 6),
        "strategy": "ensemble",
        "weights": {"mini": weights[0], "small": weights[1], "base": weights[2]},
        "timestamp": datetime.utcnow().isoformat(),
        "inference_time_ms": round((datetime.utcnow() - start_time).total_seconds() * 1000)
    }

# ============================================================
# Main prediction function (default: cascade strategy)
# ============================================================
def predict_direction(symbol: str, lookback_days: int = 30, strategy: str = "cascade") -> Dict:
    """
    Main prediction function with strategy selection.
    """
    if strategy == "cascade":
        return predict_cascade(symbol, lookback_days)
    elif strategy == "context":
        return predict_context_routed(symbol, lookback_days)
    elif strategy == "ensemble":
        return predict_ensemble(symbol, lookback_days)
    elif strategy == "mini":
        return _predict_single(symbol, predictor_mini, lookback_days, "mini", 200)
    elif strategy == "small":
        return _predict_single(symbol, predictor_small, lookback_days, "small", 50)
    elif strategy == "base":
        return _predict_single(symbol, predictor_base, lookback_days, "base", 50)
    else:
        return predict_cascade(symbol, lookback_days)

def _predict_single(symbol: str, predictor, lookback_days: int, model_name: str, lookback: int) -> Dict:
    """Helper for single model prediction"""
    start_time = datetime.utcnow()
    
    df = fetch_price_data(symbol, lookback_days)
    if df is None or len(df) < 10:
        return {"symbol": symbol, "direction": "UNKNOWN", "confidence": 0.0, "error": "Insufficient data"}
    
    try:
        x_df, x_ts, y_ts = prepare_kronos_input(df, lookback=lookback)
        result = predictor.predict(df=x_df, x_timestamp=x_ts, y_timestamp=y_ts, pred_len=1, T=1.0, top_p=0.9, sample_count=1)
        pred = result['close'].iloc[-1] if isinstance(result, pd.DataFrame) else result[0]
    except Exception as e:
        return {"symbol": symbol, "direction": "ERROR", "confidence": 0.0, "error": str(e)}
    
    direction = "UP" if pred > 0 else "DOWN"
    confidence = min(abs(pred) * 100, 0.95)
    
    return {
        "symbol": symbol,
        "direction": direction,
        "confidence": round(confidence, 3),
        "predicted_change": round(pred, 6),
        "strategy": f"single_{model_name}",
        "timestamp": datetime.utcnow().isoformat(),
        "inference_time_ms": round((datetime.utcnow() - start_time).total_seconds() * 1000)
    }

def predict_batch(symbols: List[str], strategy: str = "cascade") -> Dict:
    """Batch predictions for multiple symbols"""
    results = {}
    for symbol in symbols:
        results[symbol] = predict_direction(symbol, strategy=strategy)
    return results

print("✅ Prediction functions defined with 3-model strategies")
print("\nAvailable strategies:")
print("  - cascade:   Fast filtering (mini→small→base)")
print("  - context:   Auto-select by data length")
print("  - ensemble:  Weighted average of all models")
print("  - mini/small/base: Single model only")

## Cell 4: Start API Server
Run this to expose predictions via HTTP API

In [ ]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import nest_asyncio
import uvicorn
import threading
import time

nest_asyncio.apply()

# Create FastAPI app
app = FastAPI(
    title="Kronos Prediction API (3-Model Ensemble)",
    description="Time-series forecasting with Kronos-mini/small/base for Jasper Trades",
    version="2.0.0"
)

# Allow all origins (secure with API key in production)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class BatchRequest(BaseModel):
    symbols: List[str]
    strategy: Optional[str] = "cascade"

@app.get("/")
async def root():
    return {
        "service": "Kronos Prediction API",
        "version": "2.0.0",
        "models": ["Kronos-mini", "Kronos-small", "Kronos-base"],
        "strategies": ["cascade", "context", "ensemble", "mini", "small", "base"],
        "status": "running",
        "gpu_available": torch.cuda.is_available(),
        "models_loaded": all([model_mini, model_small, model_base])
    }

@app.get("/health")
async def health():
    return {
        "status": "healthy",
        "models_loaded": all([model_mini, model_small, model_base]),
        "gpu_available": torch.cuda.is_available(),
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"
    }

@app.get("/predict/{symbol}")
async def predict(symbol: str, strategy: str = "cascade"):
    """Get prediction for a single symbol with specified strategy"""
    return predict_direction(symbol, strategy=strategy)

@app.post("/predict/batch")
async def predict_batch_endpoint(request: BatchRequest):
    """Get predictions for multiple symbols"""
    return predict_batch(request.symbols, strategy=request.strategy)

@app.get("/test")
async def test_prediction():
    """Test prediction with a sample symbol"""
    return predict_direction("AAPL", strategy="cascade")

@app.get("/strategies")
async def list_strategies():
    """List available prediction strategies"""
    return {
        "strategies": [
            {
                "name": "cascade",
                "description": "Fast filtering: mini → small → base",
                "use_case": "Screening hundreds of pairs"
            },
            {
                "name": "context",
                "description": "Auto-select model by data length",
                "use_case": "Mixed timeframes"
            },
            {
                "name": "ensemble",
                "description": "Weighted average (20/30/50)",
                "use_case": "Maximum robustness"
            },
            {
                "name": "mini",
                "description": "Kronos-mini only (2k tokenizer, 2048 ctx)",
                "use_case": "Fast inference"
            },
            {
                "name": "small",
                "description": "Kronos-small only (base tokenizer, 512 ctx)",
                "use_case": "Balanced speed/accuracy"
            },
            {
                "name": "base",
                "description": "Kronos-base only (base tokenizer, 512 ctx)",
                "use_case": "Highest accuracy"
            }
        ]
    }

# Start server on port 8080
PORT = 8080
HOST = "0.0.0.0"

def run_server():
    uvicorn.run(app, host=HOST, port=PORT, log_level="info")

# Start server in background thread
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

print(f"\n✅ Kronos API server started on port {PORT}")
print(f"\n📡 Local URL: http://localhost:{PORT}")
print(f"\n🔧 Available strategies: cascade, context, ensemble, mini, small, base")
print(f"\n⏰ Server will keep running as long as this notebook session is active")

## Cell 5: Get Public URL
Run this AFTER Cell 4 to get the public URL for your local backend to connect to

In [ ]:
# Method 1: Use Colab's built-in proxy (recommended)
from google.colab.output import eval_js

public_url = eval_js(f"google.colab.kernel.proxyPort({PORT})")
print(f"\n🌐 Public URL: {public_url}")
print(f"\n💡 Copy this URL and paste it into your Jasper Trades backend config!")
print(f"\n📋 API Endpoints:")
print(f"  - Health:    {public_url}/health")
print(f"  - Predict:   {public_url}/predict/AAPL?strategy=cascade")
print(f"  - Batch:     {public_url}/predict/batch  (POST with JSON)")
print(f"  - Test:      {public_url}/test")
print(f"  - Strategies:{public_url}/strategies")

print(f"\n🎯 Prediction Strategies:")
print(f"  ?strategy=cascade   → Fast filtering (default)")
print(f"  ?strategy=context   → Auto-select by data length")
print(f"  ?strategy=ensemble  → Weighted average of all 3 models")
print(f"  ?strategy=mini      → Mini model only (fastest)")
print(f"  ?strategy=small     → Small model only (balanced)")
print(f"  ?strategy=base      → Base model only (most accurate)")

# Test predictions with different strategies
print(f"\n🧪 Testing predictions with different strategies...")
print("\n--- Cascade Strategy ---")
cascade_result = predict_direction("AAPL", strategy="cascade")
print(f"AAPL: {cascade_result}")

print("\n--- Ensemble Strategy ---")
ensemble_result = predict_direction("AAPL", strategy="ensemble")
print(f"AAPL: {ensemble_result}")

print("\n--- Context Strategy ---")
context_result = predict_direction("AAPL", strategy="context")
print(f"AAPL: {context_result}")

## Cell 6: Keep-Alive (Optional)
Run this to prevent Colab from disconnecting due to inactivity

In [ ]:
# JavaScript to auto-click Connect button
from IPython.display import HTML, display
import time

keep_alive_script = """
<script>
function ClickConnect(){
  console.log("Auto-connecting...");
  document.querySelector("colab-toolbar-button#connect").click();
}
setInterval(ClickConnect, 60000);
console.log("Auto-connect started. Will click Connect every 60 seconds.");
</script>
"""

display(HTML(keep_alive_script))
print("✅ Auto-connect enabled. Notebook will stay alive longer.")
print("⚠️ Note: Colab still has 12-hour session limit")

---
## Usage Instructions

### For Local Backend Connection:

1. Run all cells in order: **1 → 2 → 3 → 4 → 5**
2. Copy the Public URL from Cell 5
3. In your Jasper Trades backend config (`backend/.env` or Settings page):
   ```
   KRONOS_COLAB_URL="https://<your-colab-url>"
   USE_REMOTE_KRONOS=true
   ```

### API Endpoints:

```bash
# Health check
curl {PUBLIC_URL}/health

# Single prediction (default: cascade strategy)
curl {PUBLIC_URL}/predict/AAPL

# Single prediction with specific strategy
curl "{PUBLIC_URL}/predict/AAPL?strategy=ensemble"

# Batch predictions
curl -X POST {PUBLIC_URL}/predict/batch \
  -H "Content-Type: application/json" \
  -d '{"symbols": ["AAPL", "TSLA", "NVDA"], "strategy": "cascade"}'

# List all strategies
curl {PUBLIC_URL}/strategies
```

### Prediction Strategies:

| Strategy | Description | Use Case | Speed | Accuracy |
|----------|-------------|----------|-------|----------|
| `cascade` | Filters mini→small→base | Screening 100s of pairs | ⚡⚡⚡ | ⭐⭐⭐ |
| `context` | Auto-selects by data length | Mixed timeframes | ⚡⚡⚡ | ⭐⭐⭐ |
| `ensemble` | Weighted average (20/30/50) | Maximum robustness | ⚡ | ⭐⭐⭐⭐⭐ |
| `mini` | Mini model only (2048 ctx) | Fast inference | ⚡⚡⚡ | ⭐⭐ |
| `small` | Small model only (512 ctx) | Balanced | ⚡⚡ | ⭐⭐⭐ |
| `base` | Base model only (512 ctx) | Highest accuracy | ⚡ | ⭐⭐⭐⭐⭐ |

### Session Limits:
- **Free tier:** Up to 12 hours per session
- **GPU quota:** Varies (typically 15-20 GPU hours/day)
- **Idle timeout:** ~90 minutes of inactivity

### Tips:
- Keep this browser tab open
- Run Cell 6 to enable auto-connect
- Re-run Cell 4 if server stops
- Check `/health` endpoint to verify server is running
- Use `cascade` for screening, `ensemble` for final trades

### Memory Usage:
- **Total:** ~600 MB for all 3 models
- **GPU VRAM:** ~1 GB (well within Colab's 15 GB limit)
- **System RAM:** ~2 GB (well within Colab's 12.7 GB limit)